# 04 · Evaluate: GAN vs diffusion on unseen Kerala events

Two held-out Kerala events, never seen in training: one **landslide** (Pettimudi 2020) and
one **flood** (Kuttanad 2018). For each, four ways of producing the after image:

| Method | What it is |
|---|---|
| **pix2pix** (GAN) | trained on before/after pairs with slope, rain, soil, severity, hazard |
| **Conditional diffusion** | same data and inputs, Palette-style DDPM sampled with DDIM |
| **Identity** baseline | "nothing changes": after = before. Both models must beat it. |
| **Stable Diffusion img2img** | large pretrained model, no task training, text prompt only |

Metrics: SSIM, PSNR, FID (on RGB), generation time, and **severity consistency** (does the
measured damage follow the requested severity?). Plus a human real-vs-generated survey.
Every number in the report comes from `outputs/metrics.json`. Runtime: **GPU (T4)**.

In [ ]:
from google.colab import drive

REPO_URL = "https://github.com/muhammedshahidsds25-collab/kerala-hazard-vision-ai.git"
REPO_DIR = "/content/kerala-hazard-vision-ai"

drive.mount("/content/drive")
!git clone -q {REPO_URL} {REPO_DIR} || git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# Regular (not editable) install: an editable src-layout install is not importable
# until the runtime restarts.
!pip install -q ".[sd]"

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from kerala_hazard_vision_ai.config import load_config
from kerala_hazard_vision_ai.data.dataset import (
    TileDataset,
    list_tiles,
    load_shared_stats,
    to_reflectance,
    unnormalise_before,
)
from kerala_hazard_vision_ai.diffusion_baseline import load_pipeline, render_prompts, sd_img2img
from kerala_hazard_vision_ai.evaluate import (
    SEVERITY_CHANNEL,
    diffusion_predict,
    evaluate_identity,
    evaluate_model,
    evaluate_rgb,
    gan_predict,
    plot_comparison,
    plot_severity_sweep,
    rgb_uint8,
    save_metrics,
)
from kerala_hazard_vision_ai.human_eval import build_survey, save_rgb_pngs, score_survey
from kerala_hazard_vision_ai.models import load_generator
from kerala_hazard_vision_ai.train_diffusion import load_diffusion

cfg = load_config("configs/default.yaml")
DRIVE_ROOT = Path("/content/drive/MyDrive") / cfg.data.drive_folder
TILE_DIR = DRIVE_ROOT / "tiles"
OUT_DIR = DRIVE_ROOT / "outputs"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
GAN_DIR, DIFFUSION_DIR = DRIVE_ROOT / "checkpoints", DRIVE_ROOT / "checkpoints_diffusion"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fails loudly if the two models were trained on different tiles (e.g. after re-running 01).
stats = load_shared_stats(GAN_DIR, DIFFUSION_DIR)
test_events = [e for e in cfg.events if e.name in cfg.data.test_events]
test_sets = {e.name: TileDataset(list_tiles(TILE_DIR, [e.name]), stats, augment=False)
             for e in test_events}

generator = load_generator(GAN_DIR / "best_g.pt", cfg.model, device)
diffusion = load_diffusion(DIFFUSION_DIR / "best_ema.pt", cfg, device)
predictors = {
    "pix2pix": gan_predict(generator, device),
    "diffusion": diffusion_predict(diffusion, device, cfg.diffusion.sample_steps, cfg.train.seed),
}


def to_rgb(scaled: torch.Tensor) -> torch.Tensor:
    return rgb_uint8(to_reflectance(scaled, stats), cfg.eval.rgb_max)


def stack_tiles(ds: TileDataset, indices) -> tuple[torch.Tensor, torch.Tensor]:
    xs, ys = zip(*(ds[i] for i in indices), strict=True)
    return torch.stack(xs), torch.stack(ys)


for event in test_events:
    print(f"Test event: {event.name} ({event.hazard}), {len(test_sets[event.name])} tiles")

## 0. Survey images first (about 10 minutes)
Builds 21 images in `outputs/human_eval/`: 7 real, 7 pix2pix, 7 diffusion, shuffled, each from
a different tile of the two test events. They come first so the Google Form can be built
while the slow evaluation below runs: open `tools/survey_form.gs` (instructions at its top),
run `createSurveyForm`, and share the link with 5–10 classmates. Do **not** share `key.csv`.

In [ ]:
SURVEY_DIR = OUT_DIR / "human_eval"
pools = {"real": [], "pix2pix": [], "diffusion": []}
batch = cfg.train.batch_size
for event in test_events:
    all_x, all_y = stack_tiles(test_sets[event.name], range(len(test_sets[event.name])))
    pools["real"].append(to_rgb(all_y))
    for name, predict in predictors.items():
        pools[name].append(torch.cat(
            [to_rgb(predict(all_x[i : i + batch])) for i in range(0, len(all_x), batch)]
        ))
pngs = {name: save_rgb_pngs(torch.cat(rgb), SURVEY_DIR / "pool", name)
        for name, rgb in pools.items()}
real_pngs = pngs.pop("real")
build_survey(real_pngs, pngs, n_per_source=7, out_dir=SURVEY_DIR, seed=cfg.train.seed)
print("Survey images are ready in", SURVEY_DIR)

## 1. pix2pix vs diffusion vs identity, per test event
The diffusion model runs 50 denoising steps per image, so this cell takes longer.

In [ ]:
results = {"events": {}}
sweeps = {}
for event in test_events:
    ds = test_sets[event.name]
    scores = {"hazard": event.hazard}
    for name, predict in predictors.items():
        scores[name] = evaluate_model(predict, ds, cfg)
        sweeps[(event.name, name)] = scores[name].pop("severity_sweep")  # plotted, not tabled
    scores["identity"] = evaluate_identity(ds, cfg)
    results["events"][event.name] = scores
    display(pd.DataFrame({k: scores[k] for k in ("pix2pix", "diffusion", "identity")}).T
            .assign(event=event.name))

## 2. Figures: side by side, severity sweeps, diffusion diversity

In [ ]:
examples = {}
for event in test_events:
    ds = test_sets[event.name]
    severe_first = ds.severities.argsort()[::-1][:4].tolist()
    x, y = stack_tiles(ds, severe_first)
    before_rgb = rgb_uint8(unnormalise_before(x, stats), cfg.eval.rgb_max)
    examples[event.name] = (severe_first, x, before_rgb, to_rgb(y))
    display(plot_comparison(
        {"Before": before_rgb, "Real after": to_rgb(y),
         "pix2pix": to_rgb(predictors["pix2pix"](x)),
         "Diffusion": to_rgb(predictors["diffusion"](x))},
        FIG_DIR / f"comparison_{event.name}.png",
    ))
    for name, predict in predictors.items():
        frames = []
        for sev in cfg.demo.sweep[event.hazard]:
            swept = x[:1].clone()
            swept[:, SEVERITY_CHANNEL] = sev
            frames.append(to_rgb(predict(swept))[0])
        display(plot_severity_sweep(before_rgb[0], frames, cfg.demo.sweep[event.hazard],
                                    FIG_DIR / f"sweep_{event.name}_{name}.png"))

**Diversity:** diffusion can show several plausible outcomes for the same area and severity
(different random seeds). pix2pix always gives the same answer.

In [ ]:
steps = cfg.diffusion.sample_steps
for event in test_events:
    _, x, before_rgb, _ = examples[event.name]
    samples = {f"Seed {s}": to_rgb(diffusion_predict(diffusion, device, steps, s)(x))
               for s in range(4)}
    display(plot_comparison({"Before": before_rgb, **samples},
                            FIG_DIR / f"diffusion_diversity_{event.name}.png"))

In [ ]:
sweep_df = pd.concat(
    [pd.DataFrame(points).assign(event=event, model=model)
     for (event, model), points in sweeps.items()]
)
groups = sweep_df.groupby(["event", "model"])
fig, axes = plt.subplots(1, len(groups), figsize=(4 * len(groups), 4))
for ax, ((event, model), group) in zip(axes, groups, strict=True):
    group.boxplot(column="measured", by="requested", ax=ax)
    ax.set_title(f"{event}\n{model}")
    ax.set_xlabel("requested severity")
axes[0].set_ylabel("measured severity of generated image")
fig.suptitle("Severity control")
fig.tight_layout()
fig.savefig(FIG_DIR / "severity_consistency.png", dpi=150)

## 3. Stable Diffusion img2img (zero-shot, prompt engineering)
Same before images, 4 prompt designs from `configs/default.yaml`, filled in with each test
event's hazard. Takes ~20–30 min per event on a T4.

In [ ]:
pipe = load_pipeline(cfg.eval, device)
for event in test_events:
    ds = test_sets[event.name]
    severe_first, _, before_rgb, real_rgb = examples[event.name]
    all_x, all_y = stack_tiles(ds, range(len(ds)))
    test_before_rgb = rgb_uint8(unnormalise_before(all_x, stats), cfg.eval.rgb_max)
    test_real_rgb = to_rgb(all_y)

    sd_scores, sd_examples = {}, {}
    for prompt in render_prompts(cfg.eval.prompts, event.hazard):
        sd_rgb = sd_img2img(pipe, test_before_rgb, prompt, cfg.eval, seed=cfg.train.seed)
        sd_scores[prompt] = evaluate_rgb(sd_rgb, test_real_rgb)
        sd_examples[prompt] = sd_rgb[severe_first]
    # Chosen on the test set: this favours the baseline, which only makes our models' win harder.
    best_prompt = max(sd_scores, key=lambda p: sd_scores[p]["ssim"])
    results["events"][event.name]["stable_diffusion"] = sd_scores
    results["events"][event.name]["stable_diffusion_best_prompt"] = best_prompt
    display(pd.DataFrame(sd_scores).T.assign(event=event.name))
    display(plot_comparison(
        {"Before": before_rgb, "Real after": real_rgb,
         "Stable Diffusion": sd_examples[best_prompt]},
        FIG_DIR / f"comparison_stable_diffusion_{event.name}.png",
    ))
del pipe
torch.cuda.empty_cache()

## 4. Results table → `outputs/metrics.json`

In [ ]:
save_metrics(results, OUT_DIR / "metrics.json")
rows = []
for event_name, scores in results["events"].items():
    methods = {
        "pix2pix (GAN)": scores["pix2pix"],
        "Conditional diffusion": scores["diffusion"],
        "Identity baseline": scores["identity"],
        "Stable Diffusion (best prompt)":
            scores["stable_diffusion"][scores["stable_diffusion_best_prompt"]],
    }
    for method, metrics in methods.items():
        rows.append({"event": event_name, "hazard": scores["hazard"], "method": method, **metrics})
table = pd.DataFrame(rows)
table[["event", "hazard", "method", "n_tiles", "ssim", "psnr", "fid", "sec_per_tile"]].round(3)

## 5. Score the human evaluation survey
After collecting answers, run `exportAnswers` in `tools/survey_form.gs`: it writes
`outputs/human_eval/answers.csv` (`rater,item,answer`). Then run the setup cells (1–2) and this
cell. It reloads the saved metrics, so the models do not need to be evaluated again.
(It reads `SURVEY_DIR` set in section 0.)

In [ ]:
SURVEY_DIR = OUT_DIR / "human_eval"
results = json.loads((OUT_DIR / "metrics.json").read_text())
results["human_eval"] = score_survey(SURVEY_DIR / "answers.csv", SURVEY_DIR / "key.csv")
save_metrics(results, OUT_DIR / "metrics.json")
results["human_eval"]

Copy `outputs/metrics.json`, `outputs/figures/` and `outputs/human_eval/key.csv` +
`answers.csv` from Drive into the repo's `outputs/` folder, save this executed notebook into
`notebooks/`, then commit.